# Spark Connect from an OpenShift AI workbench

Runs against the `SparkConnect` server from `deploy/openshift-ai/20-sparkconnect.yaml`.
Workbench image: any Python 3.12 Jupyter image. Run examples 1 and 2 first so there is data to explore.

The notebook needs **no** Spark installation, JVM or cluster credentials - only the thin client.

In [1]:
# OpenShift AI workbenches install from Red Hat's Python index. In OpenShift AI 3.5 it offers
# pyspark-client up to 4.2.0, which is tested against this 4.3.0 server. With a PyPI mirror
# configured the same line installs 4.3.0. Restart the kernel after installing.
%pip install --quiet "pyspark-client>=4.2,<=4.3.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pyspark.sql import SparkSession, functions as F

NAMESPACE = os.environ.get("SPARK_CONNECT_NAMESPACE", "spark-demo")   # namespace of the SparkConnect server
BUCKET = os.environ.get("S3_BUCKET", "spark-smoke")

spark = SparkSession.builder.remote(
    f"sc://spark-connect-service.{NAMESPACE}.svc.cluster.local:15002").getOrCreate()
print("Connected. Server Spark version:", spark.version)

Connected. Server Spark version: 4.3.0


/opt/app-root/lib64/python3.12/site-packages/pyspark/sql/connect/utils.py:41: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/app-root/lib64/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


If `getOrCreate()` **hangs** rather than failing, a NetworkPolicy is dropping the traffic: the workbench pod needs the label `opendatahub.io/workbenches=true` and its namespace `opendatahub.io/dashboard=true`.

In [3]:
# Executors read the curated zone straight from S3; only the result comes back to the notebook.
txns = spark.read.parquet(f"s3a://{BUCKET}/examples/curated/transactions")
txns.printSchema()
txns.groupBy("category").agg(F.count("*").alias("txns"), F.round(F.avg("amount"), 2).alias("avg_amount")).orderBy("category").show()

root
 |-- txn_id: long (nullable = true)
 |-- account_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- risk: double (nullable = true)
 |-- dt: string (nullable = true)
 |-- category: string (nullable = true)



+---------+-----+----------+
| category| txns|avg_amount|
+---------+-----+----------+
|   dining|32989|    251.79|
|     fuel|32990|    249.82|
|  grocery|32337|    249.47|
|   online|32336|    250.47|
|   travel|32337|    250.61|
|utilities|32990|    251.84|
+---------+-----+----------+



In [4]:
# A pandas UDF defined here, executed on the cluster's Python workers (Arrow).
import pandas as pd

@F.pandas_udf("string")
def band(amount: pd.Series) -> pd.Series:
    return pd.cut(amount, [0, 50, 200, 1e9], labels=["small", "medium", "large"]).astype(str)

txns.groupBy(band("amount").alias("band")).count().toPandas()

/opt/app-root/lib64/python3.12/site-packages/pyspark/sql/pandas/functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/app-root/lib64/python3.12/site-packages/pyspark/sql/pandas/typehints.py:60: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,band,count
0,medium,58876
1,small,19320
2,large,117783


In [5]:
# Spark SQL
txns.createOrReplaceTempView("txns")
spark.sql("""
  SELECT category, account_id, round(sum(amount), 2) AS spend
  FROM txns GROUP BY category, account_id ORDER BY spend DESC LIMIT 10
""").toPandas()

,category,account_id,spend
0,utilities,4831,5319.79
1,dining,3965,5312.65
2,utilities,3347,5270.99
3,utilities,4109,5263.91
4,utilities,4959,5140.33
5,travel,3616,5113.77
6,travel,4260,5112.68
7,grocery,106,5071.37
8,grocery,4094,5055.02
9,online,82,5047.08


### Database access
JDBC credentials should come from a Secret mounted into the workbench as environment variables (data connection), never typed into a notebook.

In [6]:
jdbc = dict(url=os.environ["JDBC_URL"], user=os.environ["JDBC_USER"], password=os.environ["JDBC_PASSWORD"])
summary = spark.read.format("jdbc").options(dbtable="daily_category_summary", **jdbc).load()
summary.orderBy("category").toPandas()

,dt,category,txn_count,total_amount,avg_risk,accounts
0,2026-11-25,dining,32989,8306366.46,0.5036,2500
1,2026-11-25,fuel,32990,8241628.76,0.4996,2500
2,2026-11-25,grocery,32337,8067139.02,0.4989,2457
3,2026-11-25,online,32336,8099270.76,0.6157,2456
4,2026-11-25,travel,32337,8103816.90,0.6681,2457
5,2026-11-25,utilities,32990,8308115.50,0.5037,2500


In [7]:
spark.stop()